# 07 — Smart Execute: One Function Call from Question to Answer

**The pitch.** You have a question. You don't want to learn 85 templates, chain orchestration, or prompt engineering. You want a great answer. `smart_execute(question)` routes your question through mycontext's intelligence layer — picks the right tier, selects the right template, assembles an expert-grade prompt, calls the model — and hands you the response plus a full audit trail.

**Who this is for:** Anyone who wants the fastest path from question to quality answer.

**What you'll learn:**
- How `smart_execute` routes simple / moderate / complex questions to three different tiers
- How to inspect the `meta` dict to see *exactly* what happened
- How `smart_prompt` lets you preview the assembled prompt before spending tokens
- How `smart_generic_prompt` gives you a fast, no-compile-cost prompt for any template

**Visual map** → [assets/overview.html](assets/overview.html)

**Next:** **08** = Prompt Architect (build / parse / improve). **09** = Template selection intelligence. **10** = Quality gates.

## Research grounding

> **Routing model:** Three-tier complexity routing is grounded in *Khot et al. 2022 — "Decomposed Prompting"* — routing complex questions through specialist reasoning templates produces measurably more structured outputs than single-pass prompting. The **CAI benchmark** (mycontext's own evaluation, 200+ test cases) shows template-driven prompts produce 1.8–2.5x better outputs on analytical questions, with no benefit on simple factual queries. Three tiers exist to apply that lift only where it's warranted — keeping simple queries fast and cheap.

> **Thinking strategies in templates:** Where templates inject a reasoning strategy, they map directly to published techniques: Chain of Thought *(Wei et al. 2022)*, Tree of Thought *(Yao et al. 2023)*, Self-Reflection / Reflexion *(Shinn et al. 2023)*.

## The problem this solves

Every AI framework makes you make decisions you shouldn't have to make:

| Decision | Without mycontext | With `smart_execute` |
|----------|-------------------|----------------------|
| Which prompt template? | Browse docs, guess | Automatic |
| Simple or complex routing? | Manual triage | Three-tier auto-routing |
| Prompt quality before call? | Hope for the best | Built-in |
| Which model settings? | Tune per-call | Sensible defaults |

`smart_execute` is the **zero-decision path**. One call. We decide.

## Install and setup

In [ ]:
# %pip install -q -U "mycontext-ai>=0.10.1"
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s, raw=True)

import mycontext

md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")


## The three tiers — how routing works

Before the first call, it helps to know what's happening under the hood:

| Tier | Triggered when | What happens |
|------|---------------|---------------|
| **Raw** | Simple, factual, or conversational | Sends a clean direct prompt — no template overhead |
| **Single template** | Analytical, structured reasoning needed | Picks the best cognitive pattern from 85 templates, builds a full Context |
| **Integrated pipeline** | Multi-angle, complex question | Chains 2–5 templates, integrates them into one unified prompt |

The routing happens in `assess_complexity()` — heuristic-first, LLM-fallback.

## Step 1 — Simple question (raw tier)

In [ ]:
from mycontext.intelligence import smart_execute, assess_complexity

simple_q = "What does HTTP 429 mean?"

# Check complexity first (heuristic, no API key needed)
complexity = assess_complexity(simple_q, skip_heuristic=False)
md(f"Question: {simple_q!r}")
md(f"Routing decision: {complexity.tier}")
md(f"Reasoning: {complexity.reasoning}")


In [ ]:
import json
response, meta = smart_execute(simple_q, provider="openai")
md("=== Response ===")
md((str(response)[:500-3] + '...' if len(str(response)) > 500 else str(response)))
md("\n=== Meta ===")
_sj = json.dumps(meta, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


## Step 2 — Moderate question (single template tier)

In [ ]:
moderate_q = "What are the root causes of high customer churn in a B2B SaaS product?"

complexity = assess_complexity(moderate_q, skip_heuristic=False)
md(f"Question: {moderate_q!r}")
md(f"Routing decision: {complexity.tier}")
md(f"Reasoning: {complexity.reasoning}")


In [ ]:
import json
response, meta = smart_execute(moderate_q, provider="openai")
md("=== Response (first 800 chars) ===")
md((str(response)[:800-3] + '...' if len(str(response)) > 800 else str(response)))
md("\n=== Meta — notice: tier + template_used ===")
_sj = json.dumps(meta, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


## Step 3 — Complex question (integrated pipeline tier)

In [ ]:
complex_q = (
    "We're considering acquiring a Series B fintech company that does embedded lending. "
    "Analyze the strategic fit, key risks, and what our integration roadmap should look like "
    "for a board presentation."
)

complexity = assess_complexity(complex_q, skip_heuristic=False)
md(f"Routing decision: {complexity.tier}")
md(f"Reasoning: {complexity.reasoning}")


In [ ]:
import json
response, meta = smart_execute(complex_q, provider="openai")
md("=== Response (first 1000 chars) ===")
md((str(response)[:1000-3] + '...' if len(str(response)) > 1000 else str(response)))
md("\n=== Meta — notice: templates_used is now a list ===")
_sj = json.dumps(meta, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


## Step 4 — `smart_prompt`: see the prompt before you pay for it

`smart_execute` runs the full call. Sometimes you want to **inspect the assembled prompt first** — check its quality, tweak parameters, or use it in your own pipeline. That's `smart_prompt`.

In [ ]:
from mycontext.intelligence import smart_prompt

# Returns ComposedPrompt — no LLM call yet
composed = smart_prompt(
    "What are the risks of deploying ML models without monitoring?",
    provider="openai",
)

prompt_text = composed.to_string()
md("=== Assembled prompt (first 800 chars) ===")
md((str(prompt_text)[:800-3] + '...' if len(str(prompt_text)) > 800 else str(prompt_text)))
md(f"\nTotal length: {len(prompt_text)} chars")


In [ ]:
import json
# ComposedPrompt can export to any format
msgs = composed.to_messages()
md("OpenAI messages format:")
_sj = json.dumps(msgs, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


In [ ]:
# Score the prompt BEFORE calling the LLM
from mycontext.intelligence import QualityMetrics

if hasattr(composed, 'context') and composed.context is not None:
    qm = QualityMetrics(mode="heuristic")
    score = qm.evaluate(composed.context)
    md(f"Prompt quality score: {score.overall * 100:.0f}/100")
    for _dk, _dv in score.dimensions.items():
        md(f"  {_dk.name.title()}: {_dv * 100.0:.0f}")
else:
    md("Quality scoring available when context is attached to ComposedPrompt")


## Step 5 — `smart_generic_prompt`: instant prompt, zero compile cost

When you need a prompt fast and don't need full template compilation, `smart_generic_prompt` fills the GENERIC_PROMPT slot for the best-matching template. Heuristic-only, sub-millisecond.

In [ ]:
from mycontext.intelligence import smart_generic_prompt

result = smart_generic_prompt(
    "Analyze the competitive positioning of our product against three alternatives",
    provider=None,  # heuristic routing, no API key needed
)

md("Template selected:", getattr(result, 'template_name', 'N/A'))
prompt_str = result.to_string() if hasattr(result, 'to_string') else str(result)
md("\nGeneric prompt preview:")
md((str(prompt_str)[:600-3] + '...' if len(str(prompt_str)) > 600 else str(prompt_str)))


## The proof: same question, three paths

Let's see the same question through all three lenses — raw, single-template, and smart — and compare.

In [ ]:
from mycontext.intelligence import QualityMetrics, transform
from mycontext import Context, Guidance, Directive

question = "What risks should we consider before launching an AI feature in a regulated industry?"

# Path A: raw prompt (what most people do)
raw_ctx = Context(
    guidance=Guidance(role="assistant", rules=["Be helpful and accurate"]),
    directive=Directive(content=question),
)

# Path B: smart template selection
smart_ctx = transform(question)

qm = QualityMetrics(mode="heuristic")
raw_score = qm.evaluate(raw_ctx)
smart_score = qm.evaluate(smart_ctx)

md(f"{'Path':<30} {'Quality Score':>15} {'Dimensions'}")
md("-" * 70)
md(f"{'Raw (generic assistant)':<30} {raw_score.overall * 100:>14.0f}/100")
md(f"{'smart_execute (auto-routed)':<30} {smart_score.overall * 100:>14.0f}/100")
md(f"\nLift: +{(smart_score.overall - raw_score.overall) * 100:.0f} points from automatic template selection")


## Summary

| What you learned | API |
|-----------------|-----|
| Zero-decision execution | `smart_execute(question, provider=...)` |
| Inspect prompt before spending tokens | `smart_prompt(question)` |
| Fast heuristic-only prompt | `smart_generic_prompt(question)` |
| Understand routing logic | `assess_complexity(question)` |
| Pre-flight quality check | `QualityMetrics(mode="heuristic").evaluate(ctx)` |

**Next notebook:** [02_prompt_architect.ipynb](02_prompt_architect.ipynb) — when you want to build, parse, or improve a prompt yourself rather than letting `smart_execute` choose for you.